# C3: Operacje na atrybutach, selekcja i zapis danych
### Programowanie w GIS — Ćwiczenia

---

> **Kurs:** Programowanie w GIS (QGIS)  
> **Ćwiczenie:** C3  
> **Powiązany wykład:** W2  
> **Czas:** ~90 minut  

---

### Cele ćwiczenia

Po ukonczeniu tego cwiczenia bedziesz potrafil/a:

- programowo zaznaczac obiekty warstwy przez wyrazenie i przez zasieg przestrzenny,
- modyfikowac wartosci atrybutow w trybie edycji,
- dodawac nowe obiekty do warstwy,
- usuwac obiekty spelniajace podany warunek,
- zapisywac warstwe do nowego pliku GeoPackage.

---

### Konwencje

| Oznaczenie | Znaczenie |
|---|---|
| **Zadanie** | Polecenie do samodzielnego wykonania |
| **Wskazowka** | Pomocna podpowiedz |
| **Pytanie** | Odpowiedz wpisz w komurce ponizej |
| `# [QGIS]` | Kod do uruchomienia w Konsoli Pythona QGIS |

---

### Przygotowanie

1. Uruchom QGIS z projektem z poprzedniego cwiczenia.
2. Upewnij sie ze masz zaladowana warstwe wektorowa z polem tekstowym i numerycznym.
3. **Zrob kopie warstwy** zanim zaczniesz modyfikacje:
   prawym przyciskiem na warstwe -> `Eksportuj -> Zapisz obiekty jako...`
   -> format GeoPackage -> nazwa `cwiczenie_c3.gpkg`.
   Dalej pracuj na tej kopii, nie na oryginale.
4. Otworz Konsole Pythona i edytor skryptow.


## Spis tresci

1. [Selekcja programowa](#s1)
2. [Tryb edycji i zmiana atrybutow](#s2)
3. [Dodawanie nowych obiektow](#s3)
4. [Usuwanie obiektow](#s4)
5. [Zapis do pliku](#s5)
6. [Zadanie laczace](#s6)


---
<a id='s1'></a>

## 1. Selekcja programowa


### Zadanie 1.1

Zaladuj kopie warstwy (`cwiczenie_c3.gpkg`) do projektu jesli jeszcze tego
nie zrobiles/zrobilias. Nastepnie uruchom ponizszy kod:

```python
layer = QgsProject.instance().mapLayersByName('cwiczenie_c3')[0]

# Zaznacz obiekty przez wyrazenie
# Zmien pole i wartosc na odpowiednie dla Twojej warstwy
layer.selectByExpression('"continent" = \'Europe\'')

print(f'Zaznaczono: {layer.selectedFeatureCount()} obiektow')
print(f'Wszystkich: {layer.featureCount()}')
```

Sprawdz czy zaznaczenie jest widoczne na mapie (zolte podswietlenie).


**Pytanie 1.1** — Podaj wyrazenie ktore zaznaczy obiekty
spelniajace jednoczesnie dwa warunki: nalezace do Europy
ORAZ majace populacje powyzej 10 milionow.
Jak laczymy wiele warunkow w wyrazeniu QGIS?


In [1]:
# Twoja odpowiedz i wyrazenie:


### Zadanie 1.2

Sprawdz roznice miedzy metodami selekcji:

```python
layer = QgsProject.instance().mapLayersByName('cwiczenie_c3')[0]

# Metoda 1: selectByExpression
layer.selectByExpression('"continent" = \'Europe\'')
print(f'selectByExpression: {layer.selectedFeatureCount()}')

# Metoda 2: select przez liste id
layer.removeSelection()
ids = [ob.id() for ob in layer.getFeatures() if ob['continent'] == 'Europe']
layer.select(ids)
print(f'select(ids)        : {layer.selectedFeatureCount()}')

# Iteracja po zaznaczonych
print('\nPierwsze 3 zaznaczone:')
for i, ob in enumerate(layer.selectedFeatures()):
    if i >= 3: break
    print(f'  {ob["name"]}')

layer.removeSelection()
```


**Pytanie 1.2** — Obie metody daja ten sam wynik, ale dzialaja inaczej.
Ktora jest bardziej wydajna dla duzych warstw i dlaczego?
W jakich sytuacjach wolalbys/wolalabys uzyc metody z lista `ids`?


In [2]:
# Twoja odpowiedz:


### Zadanie 1.3 — Samodzielne

Napisz kod ktory zaznacza obiekty w **zasięgu przestrzennym** ograniczonym
do Europy Srodkowej (przyblizony bbox: lon 10–25, lat 46–55).

Wskazowka: uzyj `layer.selectByRect(QgsRectangle(...))`.
Zaimportuj `QgsRectangle` z `qgis.core`.


In [3]:
# Twoj kod:

# [QGIS]


---
<a id='s2'></a>

## 2. Tryb edycji i zmiana atrybutow


### Zadanie 2.1

Uruchom ponizszy kod i zaobserwuj co sie dzieje gdy probujemy zmienic
atrybut bez wlaczania trybu edycji:

```python
layer = QgsProject.instance().mapLayersByName('cwiczenie_c3')[0]

# Proba zmiany BEZ trybu edycji
pierwszy = next(layer.getFeatures())
fid = pierwszy.id()
idx = layer.fields().indexOf('name')  # zmien na pole ktore masz

wynik = layer.changeAttributeValue(fid, idx, 'TEST_BEZ_EDYCJI')
print(f'Wynik bez trybu edycji: {wynik}')

# Teraz z trybem edycji
layer.startEditing()
wynik2 = layer.changeAttributeValue(fid, idx, 'TEST_Z_EDYCJA')
print(f'Wynik z trybem edycji: {wynik2}')

# Sprawdz czy zmiana jest widoczna
zmieniony = layer.getFeature(fid)
print(f'Wartosc po zmianie: {zmieniony["name"]}')

# Na razie odrzuc zmiany
layer.rollBack()
print('Zmiany odrzucone.')
```


**Pytanie 2.1** — Jaka wartosc zwraca `changeAttributeValue()`
gdy tryb edycji jest wylaczony? Co oznacza ta wartosc?
Dlaczego QGIS wymaga jawnego wlaczenia trybu edycji
zamiast robic to automatycznie?


In [4]:
# Twoja odpowiedz:


### Zadanie 2.2

Napisz skrypt ktory dla **wszystkich obiektow** warstwy
dodaje nowe pole `'name_upper'` zawierajace wartosc pola `'name'`
zamieniona na wielkie litery.

Wskazowka: nowe pole mozna dodac przez:

```python
from qgis.core import QgsField
from PyQt5.QtCore import QVariant

layer.startEditing()
layer.addAttribute(QgsField('name_upper', QVariant.String))
layer.commitChanges()
```

Dopiero po dodaniu pola mozesz zapisywac do niego wartosci
(w osobnym bloku `startEditing / commitChanges`).


In [5]:
# Twoj kompletny skrypt:

# [QGIS]


**Pytanie 2.2** — Dlaczego dodanie pola i wypelnienie go wartosciami
wymagaja **dwoch osobnych** blokow `startEditing / commitChanges`
a nie jednego? Sprawdz co sie stanie gdy sprobujemy to zrobic w jednym bloku.


In [6]:
# Twoja odpowiedz:


---
<a id='s3'></a>

## 3. Dodawanie nowych obiektow


### Zadanie 3.1

Zapoznaj sie z kodem dodajacym nowy obiekt punktowy,
a nastepnie uruchom go na swojej warstwie **punktowej**
(jesli nie masz warstwy punktowej, dodaj ja przez
`Warstwa -> Nowa -> Nowa warstwa wektorowa` z geometria punktowa):

```python
from qgis.core import QgsFeature, QgsGeometry, QgsPointXY

layer = QgsProject.instance().mapLayersByName('moja_warstwa_punktowa')[0]

layer.startEditing()

nowy = QgsFeature(layer.fields())
nowy.setGeometry(QgsGeometry.fromPointXY(QgsPointXY(21.01, 52.23)))

# Ustaw atrybuty — dostosuj do pol w Twojej warstwie
nowy['name'] = 'Punkt testowy'

czy_dodano = layer.addFeature(nowy)
print(f'Dodano obiekt: {czy_dodano}')

layer.commitChanges()

# Odswicz mape
layer.triggerRepaint()
iface.mapCanvas().refresh()
print(f'Liczba obiektow po dodaniu: {layer.featureCount()}')
```


**Pytanie 3.1** — Co sie stanie jesli przy tworzeniu obiektu
przez `QgsFeature(layer.fields())` nie ustawimy geometrii (`setGeometry()`)?
Czy taki obiekt zostanie dodany do warstwy?
Sprawdz to w konsoli i opisz wynik.


In [7]:
# Twoja odpowiedz i wynik testu:


### Zadanie 3.2 — Samodzielne

Napisz skrypt ktory dodaje do warstwy punktowej
**trzy nowe punkty** rozmieszczone w roznych miejscach Polski
(mozesz wybrac Warszawa, Krakow, Wroclaw lub inne miasta).

Wymagania:

- dodaj wszystkie trzy obiekty w **jednym** bloku `startEditing / commitChanges`,
- ustaw dla kazdego punktu wartosc pola z nazwa (jesli takie pole istnieje),
- po zatwierdzeniu wyswietl nowa liczbe obiektow w warstwie.


In [8]:
# Twoj skrypt:

# [QGIS]


---
<a id='s4'></a>

## 4. Usuwanie obiektow


### Zadanie 4.1

Uruchom ponizszy kod ktory usuwa obiekty spelniajace warunek.
Dostosuj nazwe warstwy, pole i wartosc do swoich danych:

```python
layer = QgsProject.instance().mapLayersByName('cwiczenie_c3')[0]

# Zbierz id obiektow do usuniecia
do_usuniecia = [
    ob.id() for ob in layer.getFeatures()
    if ob['continent'] == 'Antarctica'  # zmien warunek
]

print(f'Obiektow do usuniecia: {len(do_usuniecia)}')
print(f'Liczba przed usuniecie: {layer.featureCount()}')

layer.startEditing()
layer.deleteFeatures(do_usuniecia)
layer.commitChanges()

print(f'Liczba po usunieciu   : {layer.featureCount()}')
```


**Pytanie 4.1** — Zauwaz ze najpierw zbieramy liste `id` do usuniecia,
a dopiero potem uruchamiamy `startEditing()` i `deleteFeatures()`.
Co by sie stalo gdybysmy probowali zbierac `id` i jednoczesnie
usuwac obiekty wewnatrz jednej petli po `layer.getFeatures()`?


In [9]:
# Twoja odpowiedz:


### Zadanie 4.2 — Samodzielne

Napisz skrypt ktory:

1. Pyta uzytkownika (przez `input()`) o nazwe pola i prog numeryczny.
2. Wyswietla ile obiektow zostanie usunietych.
3. Pyta o potwierdzenie (`'tak'` / `'nie'`).
4. Jesli uzytkownik potwierdzi — usuwa obiekty i zatwierdza zmiany.
5. Jesli nie — wyswietla komunikat i konczy bez zmian.

Wskazowka: `input()` dziala rowniez w Konsoli Pythona QGIS.


In [10]:
# Twoj skrypt:

# [QGIS]


---
<a id='s5'></a>

## 5. Zapis do pliku


### Zadanie 5.1

Zapisz aktywna warstwe do nowego pliku GeoPackage
uzywajac `QgsVectorFileWriter`:

```python
from qgis.core import QgsVectorFileWriter, QgsProject, QgsCoordinateTransformContext

layer = iface.activeLayer()
sciezka = 'C:/GIS_kurs/wynik_c3.gpkg'  # zmien sciezke

opcje = QgsVectorFileWriter.SaveVectorOptions()
opcje.driverName = 'GPKG'
opcje.fileEncoding = 'UTF-8'

blad, komunikat = QgsVectorFileWriter.writeAsVectorFormatV3(
    layer,
    sciezka,
    QgsProject.instance().transformContext(),
    opcje
)

if blad == QgsVectorFileWriter.NoError:
    print(f'Zapisano pomyslnie: {sciezka}')
else:
    print(f'Blad zapisu: {komunikat}')
```


**Pytanie 5.1** — Czym rozni sie zapis przez `QgsVectorFileWriter`
od zapisu przez prawy przycisk myszy na warstwie
(`Eksportuj -> Zapisz obiekty jako...`)?
Kiedy i dlaczego warto uzywac metody programowej?


In [11]:
# Twoja odpowiedz:


### Zadanie 5.2 — Samodzielne

Rozszerz skrypt z zadania 5.1 tak, zeby:

1. Przed zapisem reprojekcjonowal warstwe do ukladu **EPSG:2180** (Polska CS92).
2. Zapisywal tylko **zaznaczone obiekty** (jesli jakies sa zaznaczone),
   a jesli nie ma zaznaczenia — zapisywal wszystkie.

Wskazowka dla reprojekcji — ustaw w opcjach:

```python
from qgis.core import QgsCoordinateReferenceSystem
opcje.ct = QgsCoordinateTransform(
    layer.crs(),
    QgsCoordinateReferenceSystem('EPSG:2180'),
    QgsProject.instance()
)
```

Wskazowka dla tylko zaznaczonych:

```python
opcje.onlySelectedFeatures = True  # jesli sa zaznaczone
```


In [12]:
# Twoj skrypt:

# [QGIS]


---
<a id='s6'></a>

## 6. Zadanie laczace

Ponizsze zadanie wymaga polaczenia wszystkich umiejetnosci z C2 i C3.
Napisz je w edytorze skryptow QGIS.


### Zadanie 6.1 — Skrypt przetwarzania wsadowego

Napisz skrypt ktory wykonuje nastepujacy potok operacji
na warstwie `cwiczenie_c3`:

**Krok 1 — Raport wejsciowy:**
Wyswietl nazwe warstwy, liczbe obiektow i liste pol.

**Krok 2 — Dodanie pola:**
Dodaj pole tekstowe `'opis'` do warstwy.

**Krok 3 — Wypelnienie pola:**
Dla kazdego obiektu wpisz do pola `'opis'` tekst zlozony
z nazwy obiektu i jego powierzchni zaokraglonej do 2 miejsc po przecinku,
np.: `'Polska (312685.00 km2)'`.
Jesli warstwa nie jest poligonowa — wpisz dowolna kombinacje innych pol.

**Krok 4 — Selekcja:**
Zaznacz programowo 5 obiektow o najwiekszej wartosci
wybranego pola numerycznego.

**Krok 5 — Eksport:**
Zapisz tylko zaznaczone obiekty do pliku `wynik_top5.gpkg`.

**Krok 6 — Raport wyjsciowy:**
Wyswietl ile obiektow zapisano i sciezke pliku.

Wymagania:

- caly potok ma byc jednym skryptem,
- kazdemu krokowi odpowiada osobna funkcja,
- skrypt nie moze sie wylozyc jesli warstwa ma mniej niz 5 obiektow
  (obsluż ten przypadek).


In [13]:
# Twoj skrypt (wklej tutaj po napisaniu w QGIS):

# [QGIS]


**Pytanie 6.1** — W kroku 4 musisz wybrac 5 obiektow o najwyzszej wartosci.
Opisz dwa rozne sposoby jak mozna to zrobic w PyQGIS.
Ktory z nich jest bardziej wydajny dla warstwy z 1 milionem obiektow?


In [14]:
# Twoja odpowiedz:


---

## Podsumowanie C3

W tym cwiczeniu pracowales/pracowałas z:

- selekcja przez wyrazenie (`selectByExpression`) i zasieg (`selectByRect`),
- trybem edycji: `startEditing`, `commitChanges`, `rollBack`,
- zmiana atrybutow: `changeAttributeValue`, `addAttribute`,
- dodawaniem obiektow: `QgsFeature`, `QgsGeometry`, `addFeature`,
- usuwaniem obiektow: `deleteFeatures`,
- zapisem do pliku: `QgsVectorFileWriter`.

---

### Na C4

Nastepne cwiczenie poswiecone jest bibliotece GeoPandas:
wczytywanie, filtrowanie i analiza danych wektorowych poza QGIS,
w srodowisku Jupyter.

---
*Cwiczenie C3 — Programowanie w GIS (QGIS) — studia licencjackie.*
